In [6]:
from langchain_openai import ChatOpenAI
from langchain.tools import Tool
from langgraph.graph import StateGraph, END
from langchain.agents import initialize_agent
from typing import Dict, Any, Optional

# ------------------------------
# 1. LLM Setup
# ------------------------------
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# ------------------------------
# 2. JSON Schema Tool: StockPriceFetcher
# ------------------------------
def get_stock_price(ticker: str) -> str:
    prices = {"AAPL": 178.23, "GOOGL": 135.45, "TSLA": 254.78}
    price = prices.get(ticker.upper(), "Ticker not found")
    return f"The current price of {ticker.upper()} is {price}"

stock_tool_json = Tool.from_function(
    func=get_stock_price,
    name="StockPriceFetcher",
    description="Get the current stock price for a given ticker symbol.",
    return_direct=True,
    args_schema={
        "type": "object",
        "properties": {
            "ticker": {
                "type": "string",
                "description": "The stock ticker symbol, e.g., AAPL, TSLA"
            }
        },
        "required": ["ticker"]
    }
)

# ------------------------------
# 3. State Definition
# ------------------------------
class StockState(Dict):
    user_input: str
    research: Optional[str] = None
    draft: Optional[str] = None
    feedback: Optional[str] = None

# ------------------------------
# 4. Multi-Agent Nodes
# ------------------------------

# Researcher node
def researcher_node(state: StockState):
    ticker = state.get("user_input", "AAPL")
    
    # Call the JSON schema tool directly with dict
    stock_info = stock_tool_json.run({"ticker": ticker})
    
    # Ensure the result is a string
    summary = llm.invoke(f"You are Researcher. Summarize this stock info: {stock_info}").content
    
    print("\n[Researcher 📊]", summary)
    # Return only the updates to the state
    return {"research": summary}

# Writer node
def writer_node(state: StockState):
    research = state.get("research", "")
    feedback = state.get("feedback", "")
    prompt = f"You are Writer. Write a short stock report based on this info:\n{research}"
    if feedback:
        prompt += f"\nRevise based on feedback: {feedback}"
    draft = llm.invoke(prompt).content
    print("\n[Writer ✍️]", draft)
    return {"draft": draft}

# Critic node
def critic_node(state: StockState):
    draft = state.get("draft", "")
    feedback = llm.invoke(
        f"You are Critic. Review this draft:\n{draft}\n"
        f"Give constructive feedback. If it is good, reply 'APPROVED'."
    ).content
    print("\n[Critic 🧐]", feedback)
    return {"feedback": feedback}

# Conditional loop
def should_continue(state: StockState):
    feedback = state.get("feedback", "")
    if "APPROVED" in feedback.upper():
        return END
    return "writer"

# ------------------------------
# 5. LangGraph Workflow
# ------------------------------
workflow = StateGraph(StockState)
workflow.add_node("researcher", researcher_node)
workflow.add_node("writer", writer_node)
workflow.add_node("critic", critic_node)
workflow.set_entry_point("researcher")
workflow.add_edge("researcher", "writer")
workflow.add_edge("writer", "critic")
workflow.add_conditional_edges("critic", should_continue)
app = workflow.compile()

# ------------------------------
# 6. Run Example
# ------------------------------
initial_state = StockState(user_input="TSLA")
final_state = app.invoke(initial_state)

print("\n=== Final Draft ===")
print(final_state["draft"])
print("\n=== Critic Feedback ===")
print(final_state["feedback"])


[Researcher 📊] The current stock price of Tesla, Inc. (TSLA) is $254.78.

[Writer ✍️] **Stock Report: Tesla, Inc. (TSLA)**  
**Current Price:** $254.78  
**Date:** [Insert Date]

**Overview:**  
Tesla, Inc. (TSLA) is currently trading at $254.78, reflecting a stable position in the electric vehicle market amidst ongoing industry developments. The stock has shown resilience, maintaining a strong presence as one of the leading players in the EV sector.

**Market Performance:**  
In recent trading sessions, TSLA has experienced fluctuations typical of the tech and automotive sectors, influenced by broader market trends and investor sentiment. The stock's performance is closely tied to production numbers, delivery forecasts, and advancements in battery technology, which continue to drive investor interest.

**Key Drivers:**  
1. **Production and Delivery Numbers:** Tesla's ability to meet or exceed quarterly production and delivery targets remains a critical factor for its stock performan